# Partie 2 — Silver : nettoyage des données médicales avec Spark et Iceberg

Nous allons lire les trois tables **Bronze** déjà créées :

- `medical.bronze.patients`
- `medical.bronze.consultations`
- `medical.bronze.observations`

Puis nous allons produire trois tables **Silver** nettoyées :

- `medical.silver.patients`
- `medical.silver.consultations`
- `medical.silver.observations`

Les enregistrements rejetés seront conservés dans le namespace `medical.quarantine`.


In [1]:
# 1. Démarrer une nouvelle session Spark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse-Partie2-Silver")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.medical", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.medical.type", "rest")
    .config("spark.sql.catalog.medical.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.medical.warehouse", "s3://medical-warehouse/")
    .config("spark.sql.catalog.medical.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.medical.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.medical.s3.path-style-access", "true")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.ansi.enabled", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Version Spark :", spark.version)
print("Catalogue Iceberg :", spark.conf.get("spark.sql.catalog.medical.uri"))

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/19 10:17:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/10/19 10:17:20 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/10/19 10:17:20 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


Version Spark : 3.5.5
Catalogue Iceberg : http://iceberg-rest:8181


In [2]:
# 2. Lire les trois tables Bronze
noms_bronze = {
    "patients": "medical.bronze.patients",
    "consultations": "medical.bronze.consultations",
    "observations": "medical.bronze.observations"
}

for nom, table in noms_bronze.items():
    assert spark.catalog.tableExists(table), f"Table absente : {table}. Terminer la partie Bronze."
    print(nom, ":", spark.table(table).count(), "lignes")

df_patients_bronze = spark.table(noms_bronze["patients"])
df_consultations_bronze = spark.table(noms_bronze["consultations"])
df_observations_bronze = spark.table(noms_bronze["observations"])

patients : 108 lignes
consultations : 5571 lignes
observations : 68648 lignes


In [3]:
# 3. Fonctions utiles pour la partie Silver
def verifier_colonnes(df, colonnes, source):
    manquantes = [c for c in colonnes if c not in df.columns]
    assert not manquantes, f"Colonnes manquantes dans {source} : {manquantes}. Colonnes disponibles : {df.columns}"


def convertir_timestamp(nom_colonne):
    # Spark lit les dates ISO, y compris les horodatages avec fuseau horaire.
    # Ne pas couper le fuseau horaire : une date non reconnue sera rejetée.
    return F.to_timestamp(F.trim(F.col(nom_colonne)))


def creer_table_si_absente(df, nom_table):
    # En cas de deuxième exécution : conserver la table existante
    if spark.catalog.tableExists(nom_table):
        print("Table déjà existante :", nom_table, "— aucune écriture effectuée")
    else:
        df.writeTo(nom_table).using("iceberg").create()
        print("Table Iceberg créée :", nom_table)
    print("Lignes dans Iceberg :", spark.table(nom_table).count())

## Nettoyer les patients

**Règles Silver :**

1. Retirer les espaces inutiles des champs.
2. Convertir `birthdate` en type **DATE**.
3. Rejeter les lignes sans identifiant, avec date invalide ou future.
4. Garder une seule ligne par `patient_id` ; envoyer les doublons dans Quarantine.

Nous conservons le genre et la ville sans interprétation médicale.

In [4]:
# 4. Préparer et valider les patients
verifier_colonnes(
    df_patients_bronze,
    ["patient_id", "birthdate", "gender", "city"],
    "Bronze patients"
)

df_patients_prepares = (
    df_patients_bronze
    .select(
        F.trim(F.col("patient_id")).alias("patient_id"),
        F.trim(F.col("birthdate")).alias("birthdate_source"),
        F.upper(F.trim(F.col("gender"))).alias("gender"),
        F.trim(F.col("city")).alias("city")
    )
    .withColumn("birthdate", F.to_date(F.col("birthdate_source")))
    .withColumn(
        "motif_rejet",
        F.when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "Identifiant patient manquant")
         .when(F.col("birthdate").isNull(), "Date de naissance invalide")
         .when(F.col("birthdate") > F.current_date(), "Date de naissance future")
    )
)

df_patients_prepares.show(5, truncate=False)

+------------------------------------+----------------+------+----------------+----------+-----------+
|patient_id                          |birthdate_source|gender|city            |birthdate |motif_rejet|
+------------------------------------+----------------+------+----------------+----------+-----------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|1978-01-16      |M     |North Brookfield|1978-01-16|NULL       |
|8b9df7af-24b3-66ee-afeb-2734ebcbd33c|2001-02-11      |F     |Bedford         |2001-02-11|NULL       |
|65cbad6a-eb9d-420e-9bab-b4f9c6aab7db|1948-06-15      |F     |Quincy          |1948-06-15|NULL       |
|9a3814bb-f5b0-05f3-928e-fe6640d5aa2a|2015-07-19      |F     |Taunton         |2015-07-19|NULL       |
|6a703775-432b-ba7d-d53a-c4d7bb0f5b58|1995-01-23      |M     |Boston          |1995-01-23|NULL       |
+------------------------------------+----------------+------+----------------+----------+-----------+
only showing top 5 rows



In [5]:
# 5. Séparer les patients valides et les patients rejetés
patients_invalides = df_patients_prepares.filter(F.col("motif_rejet").isNotNull())

fenetre_patients = Window.partitionBy("patient_id").orderBy(
    F.col("birthdate_source"), F.col("gender"), F.col("city")
)
patients_valides_classes = (
    df_patients_prepares
    .filter(F.col("motif_rejet").isNull())
    .withColumn("rang", F.row_number().over(fenetre_patients))
)

df_patients_silver = (
    patients_valides_classes
    .filter(F.col("rang") == 1)
    .select("patient_id", "birthdate", "gender", "city")
)

patients_doublons = (
    patients_valides_classes
    .filter(F.col("rang") > 1)
    .drop("rang")
    .withColumn("motif_rejet", F.lit("Identifiant patient en double"))
)

df_patients_rejets = patients_invalides.unionByName(patients_doublons)

print("Patients Bronze :", df_patients_bronze.count())
print("Patients Silver :", df_patients_silver.count())
print("Patients rejetés :", df_patients_rejets.count())
df_patients_silver.show(5, truncate=False)

Patients Bronze : 108
Patients Silver : 108
Patients rejetés : 0
+------------------------------------+----------+------+----------+
|patient_id                          |birthdate |gender|city      |
+------------------------------------+----------+------+----------+
|00bcbfc6-c7b9-bd0e-3d95-071f9fba6321|1981-01-20|F     |Shirley   |
|029c95e2-d252-d8d3-3626-7f347182d6d7|1972-11-01|F     |Westford  |
|031ce8e2-1443-01b4-de6b-61a4084a7166|1981-01-08|M     |Canton    |
|06476266-59b1-ded7-4138-3c7543fd4981|1979-11-06|M     |Leominster|
|09c573e2-a45d-aa5e-6a17-3c682155e6e3|2008-02-12|M     |Medford   |
+------------------------------------+----------+------+----------+
only showing top 5 rows



## 5 — Créer les tables Silver et Quarantine des patients

Quarantine garde les lignes écartées avec une colonne `motif_rejet`.

In [6]:
# 6. Créer les namespaces Silver et Quarantine
spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.silver")
spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.quarantine")

spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

+----------+
|namespace |
+----------+
|bronze    |
|sandbox   |
|quarantine|
|silver    |
+----------+



In [7]:
# 7. Enregistrer les patients dans Iceberg
creer_table_si_absente(df_patients_silver, "medical.silver.patients")
creer_table_si_absente(df_patients_rejets, "medical.quarantine.patients")

Table Iceberg créée : medical.silver.patients
Lignes dans Iceberg : 108
Table Iceberg créée : medical.quarantine.patients
Lignes dans Iceberg : 0


## 6 — Nettoyer les consultations

Les consultations proviennent du fichier `encounters.csv` qui a été ingéré dans Bronze.

**Règles Silver :**

1. Renommer les champs pour obtenir des noms clairs.
2. Convertir les dates `START` et `STOP` en **TIMESTAMP**.
3. Vérifier que chaque consultation appartient à un patient Silver.
4. Rejeter les identifiants manquants, les dates invalides ou une fin antérieure au début.
5. Retirer les doublons d'identifiant.

Une consultation sans date de fin est acceptée : elle peut correspondre à un épisode non clôturé.

In [8]:
# 8. Vérifier le schéma Bronze consultations
verifier_colonnes(
    df_consultations_bronze,
    ["Id", "PATIENT", "START", "STOP", "ENCOUNTERCLASS", "CODE", "DESCRIPTION"],
    "Bronze consultations"
)

df_consultations_bronze.printSchema()
df_consultations_bronze.show(3, truncate=False)

root
 |-- Id: string (nullable = true)
 |-- START: string (nullable = true)
 |-- STOP: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- ORGANIZATION: string (nullable = true)
 |-- PROVIDER: string (nullable = true)
 |-- PAYER: string (nullable = true)
 |-- ENCOUNTERCLASS: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- BASE_ENCOUNTER_COST: string (nullable = true)
 |-- TOTAL_CLAIM_COST: string (nullable = true)
 |-- PAYER_COVERAGE: string (nullable = true)
 |-- REASONCODE: string (nullable = true)
 |-- REASONDESCRIPTION: string (nullable = true)

+------------------------------------+--------------------+--------------------+------------------------------------+------------------------------------+------------------------------------+------------------------------------+--------------+---------+------------------------------------------+-------------------+----------------+--------------+----------+-------

In [9]:
# 9. Nettoyer et contrôler les consultations
df_consultations_prepares = (
    df_consultations_bronze
    .select(
        F.trim(F.col("Id")).alias("consultation_id"),
        F.trim(F.col("PATIENT")).alias("patient_id"),
        F.trim(F.col("START")).alias("debut_source"),
        F.trim(F.col("STOP")).alias("fin_source"),
        F.trim(F.col("ENCOUNTERCLASS")).alias("categorie"),
        F.trim(F.col("CODE")).alias("code"),
        F.trim(F.col("DESCRIPTION")).alias("description")
    )
    .withColumn("date_debut", convertir_timestamp("debut_source"))
    .withColumn("date_fin", convertir_timestamp("fin_source"))
)

# Vérifier si chaque patient existe dans Silver
df_ids_patients = df_patients_silver.select("patient_id").withColumn("patient_existe", F.lit(True))
df_consultations_controlees = (
    df_consultations_prepares
    .join(df_ids_patients, on="patient_id", how="left")
    .withColumn(
        "motif_rejet",
        F.when(F.col("consultation_id").isNull() | (F.col("consultation_id") == ""), "Identifiant consultation manquant")
         .when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "Patient manquant")
         .when(F.col("patient_existe").isNull(), "Patient absent de Silver")
         .when(F.col("date_debut").isNull(), "Date de début invalide")
         .when(
             F.col("fin_source").isNotNull() & (F.col("fin_source") != "") & F.col("date_fin").isNull(),
             "Date de fin invalide"
         )
         .when(F.col("date_fin") < F.col("date_debut"), "Fin antérieure au début")
    )
    .drop("patient_existe")
)

df_consultations_controlees.show(5, truncate=False)

+------------------------------------+------------------------------------+--------------------+--------------------+---------+---------+------------------------------------------+-------------------+-------------------+-----------+
|patient_id                          |consultation_id                     |debut_source        |fin_source          |categorie|code     |description                               |date_debut         |date_fin           |motif_rejet|
+------------------------------------+------------------------------------+--------------------+--------------------+---------+---------+------------------------------------------+-------------------+-------------------+-----------+
|cc3eac4a-34a7-9a15-1522-8087135631b7|cc3eac4a-34a7-9a15-13f3-57f4a6b934f9|1996-03-11T08:35:36Z|1996-03-11T09:21:33Z|wellness |162673000|General examination of patient (procedure)|1996-03-11 08:35:36|1996-03-11 09:21:33|NULL       |
|cc3eac4a-34a7-9a15-1522-8087135631b7|cc3eac4a-34a7-9a15-1dc5-a35345

In [10]:
# 10. Séparer les consultations valides et les rejets
consultations_invalides = df_consultations_controlees.filter(F.col("motif_rejet").isNotNull())

fenetre_consultations = Window.partitionBy("consultation_id").orderBy(
    "date_debut", "patient_id", "code", "description"
)
consultations_valides_classees = (
    df_consultations_controlees
    .filter(F.col("motif_rejet").isNull())
    .withColumn("rang", F.row_number().over(fenetre_consultations))
)

df_consultations_silver = (
    consultations_valides_classees
    .filter(F.col("rang") == 1)
    .select("consultation_id", "patient_id", "date_debut", "date_fin", "categorie", "code", "description")
)

consultations_doublons = (
    consultations_valides_classees
    .filter(F.col("rang") > 1)
    .drop("rang")
    .withColumn("motif_rejet", F.lit("Identifiant consultation en double"))
)

df_consultations_rejets = consultations_invalides.unionByName(consultations_doublons)

print("Consultations Bronze :", df_consultations_bronze.count())
print("Consultations Silver :", df_consultations_silver.count())
print("Consultations rejetées :", df_consultations_rejets.count())
df_consultations_silver.show(5, truncate=False)

Consultations Bronze : 5571
Consultations Silver : 5571
Consultations rejetées : 0
+------------------------------------+------------------------------------+-------------------+-------------------+----------+---------+------------------------------------------+
|consultation_id                     |patient_id                          |date_debut         |date_fin           |categorie |code     |description                               |
+------------------------------------+------------------------------------+-------------------+-------------------+----------+---------+------------------------------------------+
|00bcbfc6-c7b9-bd0e-0c46-877b804c3c45|00bcbfc6-c7b9-bd0e-3d95-071f9fba6321|2017-12-26 01:16:49|2017-12-26 01:31:49|ambulatory|424619006|Prenatal visit (regime/therapy)           |
|00bcbfc6-c7b9-bd0e-0c7a-bfa7c60525a3|00bcbfc6-c7b9-bd0e-3d95-071f9fba6321|1999-03-16 01:16:49|1999-03-16 01:50:20|wellness  |162673000|General examination of patient (procedure)|
|00bcbfc6-c7b9-bd

In [11]:
# 11. Enregistrer les consultations dans Iceberg
creer_table_si_absente(df_consultations_silver, "medical.silver.consultations")
creer_table_si_absente(df_consultations_rejets, "medical.quarantine.consultations")

Table Iceberg créée : medical.silver.consultations
Lignes dans Iceberg : 5571
Table Iceberg créée : medical.quarantine.consultations
Lignes dans Iceberg : 0


## 7 — Nettoyer les observations médicales

Les observations ont été ingérées dans Bronze depuis `observations.jsonl`.

**Règles Silver :**

1. Convertir `DATE` en **TIMESTAMP**.
2. Vérifier que le patient existe dans Silver.
3. Rejeter une date invalide ou un code d'observation manquant.
4. Supprimer uniquement les lignes **strictement identiques**.
5. Conserver `VALUE` sous forme de texte ; créer aussi une colonne numérique si la conversion est possible.



In [12]:
# 12. Vérifier le schéma Bronze observations
verifier_colonnes(
    df_observations_bronze,
    ["DATE", "PATIENT", "ENCOUNTER", "CATEGORY", "CODE", "DESCRIPTION", "VALUE", "UNITS", "TYPE"],
    "Bronze observations"
)

df_observations_bronze.printSchema()
df_observations_bronze.show(3, truncate=False)

root
 |-- CATEGORY: string (nullable = true)
 |-- CODE: string (nullable = true)
 |-- DATE: string (nullable = true)
 |-- DESCRIPTION: string (nullable = true)
 |-- ENCOUNTER: string (nullable = true)
 |-- PATIENT: string (nullable = true)
 |-- TYPE: string (nullable = true)
 |-- UNITS: string (nullable = true)
 |-- VALUE: string (nullable = true)

+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|CATEGORY   |CODE   |DATE                |DESCRIPTION                                                  |ENCOUNTER                           |PATIENT                             |TYPE   |UNITS  |VALUE|
+-----------+-------+--------------------+-------------------------------------------------------------+------------------------------------+------------------------------------+-------+-------+-----+
|laboratory |4548-4 |2018-01-22T08:35:36Z|Hemo

In [13]:
# 13. Préparer et valider les observations
# Empreinte stable du contenu complet de chaque ligne Bronze : même ligne = même empreinte.
colonnes_bronze = [F.col(f"`{nom}`") for nom in df_observations_bronze.columns]

df_observations_prepares = (
    df_observations_bronze
    .select(
        F.sha2(F.to_json(F.struct(*colonnes_bronze)), 256).alias("observation_id"),
        F.trim(F.col("PATIENT")).alias("patient_id"),
        F.trim(F.col("ENCOUNTER")).alias("consultation_id"),
        F.trim(F.col("DATE")).alias("date_source"),
        F.trim(F.col("CATEGORY")).alias("categorie"),
        F.trim(F.col("CODE")).alias("code"),
        F.trim(F.col("DESCRIPTION")).alias("description"),
        F.trim(F.col("VALUE")).alias("valeur"),
        F.trim(F.col("UNITS")).alias("unite"),
        F.trim(F.col("TYPE")).alias("type_observation")
    )
    .withColumn("date_observation", convertir_timestamp("date_source"))
    .withColumn("valeur_numerique", F.col("valeur").cast("double"))
)

df_observations_controlees = (
    df_observations_prepares
    .join(df_ids_patients, on="patient_id", how="left")
    .withColumn(
        "motif_rejet",
        F.when(F.col("patient_id").isNull() | (F.col("patient_id") == ""), "Patient manquant")
         .when(F.col("patient_existe").isNull(), "Patient absent de Silver")
         .when(F.col("date_observation").isNull(), "Date d'observation invalide")
         .when(F.col("code").isNull() | (F.col("code") == ""), "Code d'observation manquant")
    )
    .drop("patient_existe")
)

df_observations_controlees.show(5, truncate=False)

+------------------------------------+----------------------------------------------------------------+------------------------------------+--------------------+-----------+-------+-------------------------------------------------------------+------+-------+----------------+-------------------+----------------+-----------+
|patient_id                          |observation_id                                                  |consultation_id                     |date_source         |categorie  |code   |description                                                  |valeur|unite  |type_observation|date_observation   |valeur_numerique|motif_rejet|
+------------------------------------+----------------------------------------------------------------+------------------------------------+--------------------+-----------+-------+-------------------------------------------------------------+------+-------+----------------+-------------------+----------------+-----------+
|cc3eac4a-34a7-9a15-1522-

In [14]:
# 14. Séparer les observations valides et les rejets
observations_invalides = df_observations_controlees.filter(F.col("motif_rejet").isNotNull())

fenetre_observations = Window.partitionBy("observation_id").orderBy("date_observation", "patient_id")
observations_valides_classees = (
    df_observations_controlees
    .filter(F.col("motif_rejet").isNull())
    .withColumn("rang", F.row_number().over(fenetre_observations))
)

df_observations_silver = (
    observations_valides_classees
    .filter(F.col("rang") == 1)
    .select(
        "observation_id", "patient_id", "consultation_id", "date_observation",
        "categorie", "code", "description", "valeur", "unite",
        "type_observation", "valeur_numerique"
    )
)

observations_doublons = (
    observations_valides_classees
    .filter(F.col("rang") > 1)
    .drop("rang")
    .withColumn("motif_rejet", F.lit("Ligne d'observation strictement identique"))
)

df_observations_rejets = observations_invalides.unionByName(observations_doublons)

print("Observations Bronze :", df_observations_bronze.count())
print("Observations Silver :", df_observations_silver.count())
print("Observations rejetées :", df_observations_rejets.count())
df_observations_silver.show(5, truncate=False)

Observations Bronze : 68648


Observations Silver : 68630


Observations rejetées : 18


[Stage 119:>                                                        (0 + 1) / 1]

+----------------------------------------------------------------+------------------------------------+------------------------------------+-------------------+-----------+-------+-------------------------------------------------------------------------------------------------------------+---------------------+----------------+----------------+----------------+
|observation_id                                                  |patient_id                          |consultation_id                     |date_observation   |categorie  |code   |description                                                                                                  |valeur               |unite           |type_observation|valeur_numerique|
+----------------------------------------------------------------+------------------------------------+------------------------------------+-------------------+-----------+-------+--------------------------------------------------------------------------------------------

In [15]:
# 15. Enregistrer les observations dans Iceberg
creer_table_si_absente(df_observations_silver, "medical.silver.observations")
creer_table_si_absente(df_observations_rejets, "medical.quarantine.observations")

Table Iceberg créée : medical.silver.observations
Lignes dans Iceberg : 68630


Table Iceberg créée : medical.quarantine.observations
Lignes dans Iceberg : 18


## 8 — Contrôler la qualité des données Silver

Nous vérifions que chaque ligne Bronze se retrouve **soit dans Silver, soit dans Quarantine**. Cela permet de savoir exactement combien de lignes ont été conservées ou rejetées.


In [16]:
# 16. Comparer les quantités Bronze, Silver et Quarantine
jeux = ["patients", "consultations", "observations"]

for nom in jeux:
    n_bronze = spark.table(f"medical.bronze.{nom}").count()
    n_silver = spark.table(f"medical.silver.{nom}").count()
    n_rejets = spark.table(f"medical.quarantine.{nom}").count()

    print(f"{nom:15} Bronze={n_bronze}  Silver={n_silver}  Rejets={n_rejets}")
    assert n_bronze == n_silver + n_rejets, (
        f"Comptage incohérent pour {nom}. Vérifier les tables existantes et les traitements."
    )

print("Contrôle des quantités : OK")

patients        Bronze=108  Silver=108  Rejets=0
consultations   Bronze=5571  Silver=5571  Rejets=0
observations    Bronze=68648  Silver=68630  Rejets=18
Contrôle des quantités : OK


In [17]:
# 17. Vérifier les identifiants et les relations entre tables
spark.sql("""
    SELECT COUNT(*) AS nb_patients,
           COUNT(DISTINCT patient_id) AS nb_ids_distincts
    FROM medical.silver.patients
""").show()

spark.sql("""
    SELECT COUNT(*) AS consultations_sans_patient
    FROM medical.silver.consultations c
    LEFT ANTI JOIN medical.silver.patients p
      ON c.patient_id = p.patient_id
""").show()

spark.sql("""
    SELECT COUNT(*) AS observations_sans_patient
    FROM medical.silver.observations o
    LEFT ANTI JOIN medical.silver.patients p
      ON o.patient_id = p.patient_id
""").show()

# Attendu : identifiants patients uniques, 0 consultation et 0 observation sans patient.

+-----------+----------------+
|nb_patients|nb_ids_distincts|
+-----------+----------------+
|        108|             108|
+-----------+----------------+

+--------------------------+
|consultations_sans_patient|
+--------------------------+
|                         0|
+--------------------------+

+-------------------------+
|observations_sans_patient|
+-------------------------+
|                        0|
+-------------------------+



## 9 — Explorer les résultats Silver en SQL

Les dates ne sont plus des textes : Spark peut les manipuler comme de vrais types `DATE` et `TIMESTAMP`.

In [18]:
# 18. Examiner les types et quelques enregistrements Silver
for table in ["patients", "consultations", "observations"]:
    nom = f"medical.silver.{table}"
    print("\n===", nom, "===")
    spark.table(nom).printSchema()
    spark.table(nom).show(3, truncate=False)


=== medical.silver.patients ===
root
 |-- patient_id: string (nullable = true)
 |-- birthdate: date (nullable = true)
 |-- gender: string (nullable = true)
 |-- city: string (nullable = true)

+------------------------------------+----------+------+--------+
|patient_id                          |birthdate |gender|city    |
+------------------------------------+----------+------+--------+
|00bcbfc6-c7b9-bd0e-3d95-071f9fba6321|1981-01-20|F     |Shirley |
|029c95e2-d252-d8d3-3626-7f347182d6d7|1972-11-01|F     |Westford|
|031ce8e2-1443-01b4-de6b-61a4084a7166|1981-01-08|M     |Canton  |
+------------------------------------+----------+------+--------+
only showing top 3 rows


=== medical.silver.consultations ===
root
 |-- consultation_id: string (nullable = true)
 |-- patient_id: string (nullable = true)
 |-- date_debut: timestamp (nullable = true)
 |-- date_fin: timestamp (nullable = true)
 |-- categorie: string (nullable = true)
 |-- code: string (nullable = true)
 |-- description: stri

In [19]:
# 19. Exemple SQL simple : répartition des patients par genre
spark.sql("""
    SELECT gender, COUNT(*) AS nombre_patients
    FROM medical.silver.patients
    GROUP BY gender
    ORDER BY nombre_patients DESC
""").show()

+------+---------------+
|gender|nombre_patients|
+------+---------------+
|     M|             57|
|     F|             51|
+------+---------------+



## 10 — Vérifier les fichiers Iceberg et les snapshots

In [20]:
# 20. Voir les fichiers Parquet référencés par Iceberg
spark.sql("""
    SELECT file_path, record_count, file_size_in_bytes
    FROM medical.silver.patients.files
""").show(truncate=False)

+---------------------------------------------------------------------------------------------------------+------------+------------------+
|file_path                                                                                                |record_count|file_size_in_bytes|
+---------------------------------------------------------------------------------------------------------+------------+------------------+
|s3://medical-warehouse/silver/patients/data/00000-19-72d70ec6-f3b3-4229-b57d-6e7b7c3ac956-0-00001.parquet|108         |4514              |
+---------------------------------------------------------------------------------------------------------+------------+------------------+



In [21]:
# 21. Voir le snapshot des patients Silver
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM medical.silver.patients.snapshots
""").show(truncate=False)

+-------------------+-----------------------+---------+
|snapshot_id        |committed_at           |operation|
+-------------------+-----------------------+---------+
|2539911349681929581|2026-10-19 10:17:31.764|append   |
+-------------------+-----------------------+---------+

